In [ ]:
import json

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.quiver import Quiver
from matplotlib.transforms import Affine2D
from mpl_toolkits.axes_grid1 import Divider, Size
import numpy as np
from datafold import TSCDataFrame

from dynamics_analysis_with_koopman.config.configs import PROJECT_ROOT
from dynamics_analysis_with_koopman.utils.checkpoint_registry import (
    checkpoint_path_for,
    load_run_from_checkpoint,
    load_train_tsc_from_checkpoint,
    load_original_ts_from_checkpoint,
)
from dynamics_analysis_with_koopman.plots.grid import (
    _create_ts_plot_from_df,
    _parse_measurement_cell,
    _compute_grid_rotation,
)
from dynamics_analysis_with_koopman.plots.layout import add_colorbar_next_to_grid
from dynamics_analysis_with_koopman.plots.config import DEFAULT_RIGHT_MARGIN_COLORBAR
from dynamics_analysis_with_koopman.plots.io_utils import save_figure

# importing from dynamics_analysis_with_koopman.plots runs setup_seaborn_style()
# reset to plain matplotlib defaults so this notebook's own plots
plt.rcdefaults()

TEMPORAL_RESOLUTION = 0.2
REMOVE_SPECTRAL_POLLUTION = False
FIGURE_FORMAT = 'svg'  # change this alone (e.g. to 'eps' or 'png') to switch the saved plot format

PLOTS_ROOT = PROJECT_ROOT / 'papers' / 'clustering'
PLOTS_DIR = 'ground_truth'

# absolute width (in cm) and font size (in points) used for grid plots that will
# be placed side by side in the paper, so scenarios with a different number of
# grid cells still end up the same physical size and font size
PAPER_GRID_WIDTH_CM = 5.0
PAPER_FONT_SIZE = 14

# small fixed breathing-room margin (in inches) used between measured text
# elements (titles, tick labels, legend, ...) when computing paddings below;
# actual paddings are measured from the real rendered text, so they scale
# automatically with font size
LAYOUT_MARGIN_IN = 0.12

# fixed width (in inches) of each grid's colorbar, and the gap between a grid
# and its own colorbar
CBAR_WIDTH_IN = 0.14
CBAR_GAP_IN = LAYOUT_MARGIN_IN

# extra breathing room (in inches) added between one grid's colorbar and the
# next grid's own y-tick labels, on top of the usual margins
CBAR_TO_NEXT_GRID_EXTRA_GAP_IN = 0.2

TS_ROW_HEIGHT_CM = 3.0

RUN_REGISTRY = PROJECT_ROOT / 'papers' / 'clustering' / 'juelich_run_registry.json'
with open(RUN_REGISTRY) as f:
    registry = json.load(f)

In [ ]:
# Helper functions
def _annotate_measurement_cell(axs, measurement_cell, style):
    """Draw an orange box around `measurement_cell` on both the density (imshow)
    and velocity (quiver) axes, matching whatever rotation `style` applies to
    the grid itself."""
    x_idx, y_idx = _parse_measurement_cell(measurement_cell)
    box_kwargs = dict(facecolor='none', linewidth=1.5)

    x_length, y_length = style.grid_shape[1], style.grid_shape[0]

    # imshow grid: cell (row=y_idx, col=x_idx) spans [x_idx, x_idx+1] x [y_idx, y_idx+1]
    density_patch = Rectangle((x_idx, y_idx), 1, 1, edgecolor='orange', **box_kwargs)
    # quiver grid: points are placed at integer coordinates (x_idx, y_idx), 1 unit apart
    velocity_patch = Rectangle((x_idx - 0.5, y_idx - 0.5), 1, 1, edgecolor='black', **box_kwargs)

    if style.rotation != 0:
        # same rotation + re-centering transform _draw_scalar_grid/_draw_vector_grid
        # apply to the grid itself, so the box tracks the rotated cell exactly
        _, center, shift, _ = _compute_grid_rotation(x_length, y_length, style.rotation)
        transform = Affine2D().rotate_deg_around(*center, -style.rotation).translate(*shift)
        density_patch.set_transform(transform + axs[0].transData)
        velocity_patch.set_transform(transform + axs[1].transData)

    axs[0].add_patch(density_patch)
    axs[1].add_patch(velocity_patch)


def _bbox_height_in(fig, renderer, artist):
    return artist.get_window_extent(renderer=renderer).transformed(fig.dpi_scale_trans.inverted()).height


def _bbox_width_in(fig, renderer, artist):
    return artist.get_window_extent(renderer=renderer).transformed(fig.dpi_scale_trans.inverted()).width


def _left_overhang_in(fig, renderer, ax):
    """How far (inches) the leftmost y-tick label extends past `ax`'s own left edge.

    Using the label's rendered position (rather than just its own text width)
    captures whatever tick length / tick pad the axes applies -- a rotated grid
    gets extra tick pad from `format_grid_axes`, which a plain text-width
    measurement misses, clipping the label off the saved figure.
    """
    labels = ax.get_yticklabels()
    if not labels:
        return 0.0
    ax_left_px = ax.get_window_extent(renderer=renderer).x0
    label_left_px = min(t.get_window_extent(renderer=renderer).x0 for t in labels)
    return max(ax_left_px - label_left_px, 0.0) / fig.dpi


def _pin_suptitle_top(fig, fig_height_in):
    """
    `fig.suptitle` positions itself at a fixed figure-fraction (~0.98) by default,
    independent of any axes layout -- pin it explicitly to the top of the figure
    so it always lands inside the padding reserved for it, at any font size.
    """
    if fig._suptitle is not None:
        fig._suptitle.set_verticalalignment('top')
        fig._suptitle.set_y((fig_height_in - LAYOUT_MARGIN_IN) / fig_height_in)


def _fix_grid_axes_size(fig, axs, width_in, height_in, bottom_labels=None, colorbars=None):
    """
    Forces each axes in `axs` (arranged in a single row) to have an identical,
    absolute plot-area size of `width_in` x `height_in` inches, independent of
    the number of grid cells. All paddings (for y-tick labels, the suptitle, the
    top x-tick labels from `ax.xaxis.tick_top()`, one `bottom_labels` text artist
    per axes placed underneath it, and -- if given -- one `colorbars` axes per
    grid placed to its right) are measured from the actual rendered text, so
    this scales correctly with font size. This overrides any prior
    tight_layout-based positioning, since the locator set here governs the
    final draw position.
    """
    n = len(axs)

    # pass 1: left/gap paddings only depend on y-tick label widths (and, if
    # present, the previous grid's colorbar tick label widths), which are
    # independent of the figure's width -- fix the figure's final width first
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    def cbar_label_w(i):
        if colorbars is None:
            return 0.0
        return max((_bbox_width_in(fig, renderer, t) for t in colorbars[i].get_yticklabels()), default=0.0)

    pad_left_in = _left_overhang_in(fig, renderer, axs[0]) + LAYOUT_MARGIN_IN
    gaps_in = [
        cbar_label_w(i)
        + LAYOUT_MARGIN_IN
        + (CBAR_TO_NEXT_GRID_EXTRA_GAP_IN if colorbars is not None else 0.0)
        + _left_overhang_in(fig, renderer, axs[i + 1])
        + LAYOUT_MARGIN_IN
        for i in range(n - 1)
    ]
    pad_right_in = LAYOUT_MARGIN_IN + (cbar_label_w(n - 1) + LAYOUT_MARGIN_IN if colorbars is not None else 0.0)

    horiz = [Size.Fixed(pad_left_in)]
    main_nx = []
    cbar_nx = []
    for i in range(n):
        main_nx.append(len(horiz))
        horiz.append(Size.Fixed(width_in))
        if colorbars is not None:
            horiz.append(Size.Fixed(CBAR_GAP_IN))
            cbar_nx.append(len(horiz))
            horiz.append(Size.Fixed(CBAR_WIDTH_IN))
        if i < n - 1:
            horiz.append(Size.Fixed(gaps_in[i]))
    horiz.append(Size.Fixed(pad_right_in))
    fig_width_in = sum(size.fixed_size for size in horiz)

    # pass 2: apply the final width, then measure vertical paddings -- a
    # wrap=True suptitle reflows based on the actual figure width, so its
    # height can only be measured correctly once the width is final
    fig.set_size_inches(fig_width_in, fig.get_size_inches()[1])
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    suptitle_h = _bbox_height_in(fig, renderer, fig._suptitle) if fig._suptitle is not None else 0.0
    xtick_h = max((_bbox_height_in(fig, renderer, t) for ax in axs for t in ax.get_xticklabels()), default=0.0)
    pad_top_in = suptitle_h + LAYOUT_MARGIN_IN + xtick_h + LAYOUT_MARGIN_IN

    pad_bottom_in = LAYOUT_MARGIN_IN
    if bottom_labels is not None:
        label_h = max((_bbox_height_in(fig, renderer, lbl) for lbl in bottom_labels), default=0.0)
        pad_bottom_in += label_h + LAYOUT_MARGIN_IN

    vert = [Size.Fixed(pad_bottom_in), Size.Fixed(height_in), Size.Fixed(pad_top_in)]

    divider = Divider(fig, (0, 0, 1, 1), horiz, vert, aspect=False)
    for i, ax in enumerate(axs):
        ax.set_axes_locator(divider.new_locator(nx=main_nx[i], ny=1))
    if colorbars is not None:
        for i, cbar_ax in enumerate(colorbars):
            cbar_ax.set_axes_locator(divider.new_locator(nx=cbar_nx[i], ny=1))

    fig_height_in = sum(size.fixed_size for size in vert)
    fig.set_size_inches(fig_width_in, fig_height_in)
    _pin_suptitle_top(fig, fig_height_in)

    if bottom_labels is not None:
        # center each label under its own axes column (not its colorbar), using
        # the same left/width/gap layout computed above
        x_in = pad_left_in
        for i, label in enumerate(bottom_labels):
            center_x_in = x_in + width_in / 2
            label.set_verticalalignment('bottom')
            label.set_position((center_x_in / fig_width_in, LAYOUT_MARGIN_IN / fig_height_in))
            x_in += width_in
            if colorbars is not None:
                x_in += CBAR_GAP_IN + CBAR_WIDTH_IN
            if i < n - 1:
                x_in += gaps_in[i]


def _plot_ground_truth_grids(
    tsc,
    series_id,
    scenario,
    timestep,
    measurement_cell,
    style,
    grid_width_cm=PAPER_GRID_WIDTH_CM,
    font_size=PAPER_FONT_SIZE,
):
    """
    Plots the ground truth density (image) and velocity (quiver) grids for a single
    series id at a single timestep, reusing the standard `_create_ts_plot_from_df`
    grid plotting functionality from `dynamics_analysis_with_koopman.plots.grid`.
    Optionally annotates the measurement cell (e.g. '(13_4)') on both grids. Each
    grid has no title; instead a "<Observable> at t = X.Xs" label is shown
    underneath it, and its own colorbar is shown to its right (a cyclic angle
    scale for velocity, matching the quiver coloring `_plot_single_grid` uses
    elsewhere).

    Each grid's plot area is fixed to `grid_width_cm` wide (height follows from the
    grid's data aspect ratio), and all text uses `font_size`, so that grids for
    scenarios with a different number of cells still end up the same physical size
    and font size once placed side by side in the paper.

    Saved under `<style.path>/ground_truth/` as `<style.fig_filetype>`.
    """
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))

    bottom_labels = []
    colorbar_axes = []
    for observable_type, ax in [('density', axs[0]), ('velocity', axs[1])]:
        vis_columns = [column for column in tsc.columns if observable_type in column]
        dataframe = tsc[vis_columns].select_time_values(np.array([timestep])).loc[series_id]

        _, _, im = _create_ts_plot_from_df(
            dataframe,
            [timestep],
            style,
            plot=(fig, np.array([ax])),
            remove_title=True,
        )
        ax.tick_params(labelsize=font_size)

        label_text = f'{observable_type.capitalize()} at t = {timestep * style.temporal_resolution:.1f}s'
        bottom_labels.append(fig.text(0.0, 0.0, label_text, ha='center', fontsize=font_size))

        # placeholder position/size -- _fix_grid_axes_size positions it for real
        cbar_ax = fig.add_axes((0.0, 0.0, 0.01, 0.01))
        if isinstance(im, Quiver):
            im.set_clim(0.0, 1.0)
            cbar = fig.colorbar(im, cax=cbar_ax, ticks=[0.0, 0.5, 1.0])
            cbar.ax.set_yticklabels(['$-\\pi$', '0', '$\\pi$'])
        else:
            fig.colorbar(im, cax=cbar_ax)
        cbar_ax.tick_params(labelsize=font_size)
        colorbar_axes.append(cbar_ax)

    if measurement_cell is not None:
        _annotate_measurement_cell(axs, measurement_cell, style)

    fig.suptitle(f'Ground truth for {scenario}, series {series_id}\n', fontsize=font_size, wrap=True)

    # fix the physical size of both grids to `grid_width_cm`, with the height
    # following the grid's data aspect ratio (rows, cols swapped for a 90/270 rotation)
    rows, cols = style.grid_shape
    if style.rotation in (90, 270):
        rows, cols = cols, rows
    resolution_x, resolution_y = style.grid_resolution
    width_in = grid_width_cm / 2.54
    height_in = width_in * (rows * resolution_y) / (cols * resolution_x)
    _fix_grid_axes_size(fig, axs, width_in, height_in, bottom_labels=bottom_labels, colorbars=colorbar_axes)

    cell_suffix = f'_{measurement_cell.replace("(", "").replace(")", "")}' if measurement_cell is not None else ''
    name = f'{scenario}_ground_truth_grid_{series_id}_t{timestep}{cell_suffix}'
    save_figure(fig, style, PLOTS_DIR, name, close_after=False, bbox_inches=None)
    print(f'Stored the ground truth grid plot to {style.path / PLOTS_DIR / f"{name}.{style.fig_filetype}"}')

    return fig, axs


def _plot_ground_truth_row(tsc, series_id, scenario, observable_type, timesteps, style):
    """
    Plots a single row of ground truth grid panels for `observable_type`, one
    panel per entry in `timesteps`, with one shared colorbar to the right --
    i.e. the "True values" row of `visualize_error_grid`, reproduced on its
    own without the error/prediction rows.

    Saved under `<style.path>/ground_truth/` as `<style.fig_filetype>`.
    """
    vis_columns = [column for column in tsc.columns if observable_type in column]
    dataframe = tsc[vis_columns].select_time_values(np.array(timesteps)).loc[series_id]

    fig, axs, im = _create_ts_plot_from_df(
        dataframe,
        timesteps,
        style,
        title=f'{observable_type.capitalize()} ground truth for {scenario}, series {series_id}',
    )

    # Make room on the right BEFORE adding the colorbar, then add it last so
    # it is placed against the final margins (matches visualize_grid_for_series_id)
    fig.subplots_adjust(right=DEFAULT_RIGHT_MARGIN_COLORBAR)
    add_colorbar_next_to_grid(fig, axs[-1], im, label=observable_type)

    name = f'{scenario}_ground_truth_row_{observable_type}_{series_id}'
    save_figure(fig, style, PLOTS_DIR, name, close_after=False, bbox_inches=None)
    print(f'Stored the ground truth row plot to {style.path / PLOTS_DIR / f"{name}.{style.fig_filetype}"}')

    return fig, axs


def _fix_column_axes_size(fig, axs, width_in, height_in, legend):
    """
    Forces each axes in `axs` (stacked in a single column, top to bottom, sharing
    the x-axis) to have an identical, absolute size of `width_in` x `height_in`
    inches, independent of scenario-specific content. All paddings (the
    suptitle, each row's own title, the shared x-axis label + tick labels, and
    -- if present -- the legend) are measured from the actual rendered text, so
    this scales correctly with font size and with however many legend entries
    there are. This overrides any prior tight_layout-based positioning, since
    the locator set here governs the final draw position.
    """
    n = len(axs)

    # pass 1: left padding only depends on y-tick/y-label widths, which are
    # independent of the figure's width -- fix the figure's final width first
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    ytick_w = max((_bbox_width_in(fig, renderer, t) for ax in axs for t in ax.get_yticklabels()), default=0.0)
    ylabel_w = max((_bbox_width_in(fig, renderer, ax.yaxis.label) for ax in axs), default=0.0)
    # extra margin beyond the usual 2x: fig.align_ylabels() (called after this
    # function) re-centers each y-label on the same vertical line, which can push
    # a label slightly further left than its own individually-measured width, and
    # without this room labels get clipped off the left edge of the saved SVG
    pad_left_in = ytick_w + ylabel_w + 3 * LAYOUT_MARGIN_IN
    pad_right_in = LAYOUT_MARGIN_IN
    fig_width_in = pad_left_in + width_in + pad_right_in

    # pass 2: apply the final width, then measure vertical paddings -- a
    # wrap=True suptitle reflows based on the actual figure width, so its
    # height can only be measured correctly once the width is final
    fig.set_size_inches(fig_width_in, fig.get_size_inches()[1])
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    suptitle_h = _bbox_height_in(fig, renderer, fig._suptitle) if fig._suptitle is not None else 0.0
    top_title_h = _bbox_height_in(fig, renderer, axs[0].title)
    row_title_heights = [_bbox_height_in(fig, renderer, ax.title) for ax in axs[1:]]
    # every axes shows its own x-tick labels and its own x-axis label (axs are
    # not linked via sharex), so the gap above a row's title must also clear
    # both of those from the row above it, not just that row's own title
    upper_xtick_h = max(
        (_bbox_height_in(fig, renderer, t) for ax in axs[:-1] for t in ax.get_xticklabels()), default=0.0
    )
    upper_xlabel_h = max((_bbox_height_in(fig, renderer, ax.xaxis.label) for ax in axs[:-1]), default=0.0)
    vgap_in = (
        upper_xtick_h
        + LAYOUT_MARGIN_IN
        + upper_xlabel_h
        + LAYOUT_MARGIN_IN
        + (max(row_title_heights) if row_title_heights else 0.0)
        + 2 * LAYOUT_MARGIN_IN
    )
    pad_top_in = suptitle_h + 2 * LAYOUT_MARGIN_IN + top_title_h

    tick_h = max((_bbox_height_in(fig, renderer, t) for t in axs[-1].get_xticklabels()), default=0.0)
    xlabel_h = _bbox_height_in(fig, renderer, axs[-1].xaxis.label)
    pad_bottom_in = tick_h + 2 * LAYOUT_MARGIN_IN + xlabel_h
    if legend is not None:
        legend_h = _bbox_height_in(fig, renderer, legend)
        pad_bottom_in += 2 * LAYOUT_MARGIN_IN + legend_h

    horiz = [Size.Fixed(pad_left_in), Size.Fixed(width_in), Size.Fixed(pad_right_in)]
    vert = [Size.Fixed(pad_bottom_in)]
    for i in range(n):
        vert.append(Size.Fixed(height_in))
        vert.append(Size.Fixed(vgap_in if i < n - 1 else pad_top_in))

    divider = Divider(fig, (0, 0, 1, 1), horiz, vert, aspect=False)
    for i, ax in enumerate(axs):
        # axs is ordered top to bottom; Divider's ny counts cells from the bottom
        row_from_bottom = n - 1 - i
        ny = 1 + 2 * row_from_bottom
        ax.set_axes_locator(divider.new_locator(nx=1, ny=ny))

    fig_height_in = sum(size.fixed_size for size in vert)
    fig.set_size_inches(fig_width_in, fig_height_in)
    _pin_suptitle_top(fig, fig_height_in)

In [ ]:
## Window predictions for spot check reconstructions
# time series for method overview
validation_time_series_overview = {
    '2017simeB': [10015031503, '(22_7)'],
}

MAX_TRAIN_TIMESTEP = 18
Y_LIM = [-0.05, 0.35]  # velocity_x
X_LIM = [0, 18]

for scenario, (series_id, measurement_cell) in validation_time_series_overview.items():
    print(f'Process scenario {scenario}')

    # load the model from the checkpoint
    edmd, run_config, run_metrics, style = load_run_from_checkpoint(experiment=scenario, registry=registry)
    style = style.with_(path=PLOTS_ROOT, fig_filetype=FIGURE_FORMAT, dpi=300)

    train_tsc = load_train_tsc_from_checkpoint(checkpoint_path_for(experiment=scenario, registry=registry))

    # create a figure per series
    fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(3.5, 3.5), sharex=True)
    axs = axs if isinstance(axs, np.ndarray) else np.array([axs])

    true_data = train_tsc.loc[series_id]

    for observable, ax in zip(['density'], axs):
        # add some visualization timestep lines
        visualization_timesteps = [0, 8 / TEMPORAL_RESOLUTION, 16 / TEMPORAL_RESOLUTION]
        for timestep in visualization_timesteps:
            ax.axvline(timestep * TEMPORAL_RESOLUTION, color='black', linestyle='--', alpha=0.5, linewidth=0.7)

        # visualize the time series
        cell_str = f'{measurement_cell}_{observable}'
        true_time_values = TSCDataFrame.from_single_timeseries(train_tsc.loc[series_id]).time_values()
        ax.plot(true_time_values * TEMPORAL_RESOLUTION, true_data[cell_str], linestyle='solid', color='#3B76AF')

        # configure the plot
        ax.set_ylabel(observable.replace('_', ' '))

    # configure full figure
    ax.set_xlabel('Time in seconds')
    ax.set_ylim(Y_LIM[0], Y_LIM[1])
    ax.set_xlim(X_LIM[0], X_LIM[1])
    ax.grid()
    fig.suptitle(
        f'Ground truth {observable.replace("_", " ")} for {scenario} in {measurement_cell}, series id {series_id} '
    )
    fig.tight_layout(h_pad=6)
    fig.subplots_adjust(bottom=0.30)
    # ax.legend(loc='upper center', bbox_to_anchor=(0.45, -0.27), ncol=2, fontsize=10)

    # save the plot
    name = f'ground_truth_timeseries_{scenario}_{series_id}_{measurement_cell.replace("(", "").replace(")", "")}'
    save_figure(fig, style, PLOTS_DIR, name, close_after=False, bbox_inches='tight', pad_inches=0)
    print(f'Stored the ground truth timeseries plot to {style.path / PLOTS_DIR / f"{name}.{style.fig_filetype}"}')
    plt.show()

In [ ]:
# for the in depth examples --> no measurement cell
# use the cropped time series (i.e. from learning run)
VECTOR_SCALING = 0.1

validation_time_series = {
    ## for method overview
    '2017simeB': [10015031503, None, [0, 8, 16, 24, 32]],
    # # for lane formations
    # '2013bidirectionalB':  [1004000505, None, 36],
    # '2013crossing120A':    [10012010505, None, 40],
    # for instructions
    '2013crossing90D': [10090488, None, 12],
    '2013crossing90E': [10090566, None, 12],
}

## Plot the window predictions for each scenario
# get the window predictions
for scenario, (series_id, measurement_cell, time_in_s) in validation_time_series.items():
    # load the model + run config/metrics/style from the checkpoint
    edmd, run_config, run_metrics, style = load_run_from_checkpoint(experiment=scenario, registry=registry)
    style = style.with_(path=PLOTS_ROOT, fig_filetype=FIGURE_FORMAT, dpi=300, vector_scaling=VECTOR_SCALING)

    train_tsc, _ = load_train_tsc_from_checkpoint(
        checkpoint_path_for(experiment=scenario, registry=registry), with_predictions=True
    )

    print(f'\n\n\nProcess scenario {scenario}')
    # plot the ground truth density and velocity grids for a single visualization step
    true_time_values = TSCDataFrame.from_single_timeseries(train_tsc.loc[series_id]).time_values()
    time_in_s = time_in_s if isinstance(time_in_s, list) else [time_in_s]
    ground_truth_timesteps = [true_time_values[int(t / style.temporal_resolution)] for t in time_in_s]

    if len(ground_truth_timesteps) > 1:
        # reproduce just the first ("True values") row of the error grid plot --
        # one row of density panels, all timesteps side by side -- instead of a
        # separate density + velocity grid per timestep
        _plot_ground_truth_row(train_tsc, series_id, scenario, 'density', ground_truth_timesteps, style)
    else:
        _plot_ground_truth_grids(train_tsc, series_id, scenario, ground_truth_timesteps[0], measurement_cell, style)

In [ ]:
## Plot the ground truth single density and velocity grids, with highlighted measurement cell
# use the time series before applying the cut-off, i.e. from the dmd-preprocessing run
from dynamics_analysis_with_koopman.config.configs import FullConfig
from dynamics_analysis_with_koopman.pipeline.results import RunMetrics
from dynamics_analysis_with_koopman.plots.style import PlotStyle

VECTOR_SCALING = 0.1

validation_time_series = {
    # for spot-check
    '2009tjunction': [100240150240240, '(13_4)', 12],
    '2009unidirOpenC': [100050180180180, '(7_18)', 16],
    '2013crossing90E': [10090511, '(13_22)', 60],
    '2013crossing120C': [10012030707, '(8_14)', 24],
    '2013unidirectionalC': [1005000101, '(10_7)', 32],
    '2022wheelchairU': [1001100100, '(10_7)', 8],
}

for scenario, (series_id, measurement_cell, time_in_s) in validation_time_series.items():
    # load run config/metrics/style from the preprocessing checkpoint
    checkpoint_path = checkpoint_path_for(experiment=scenario, registry=registry, run_selector='preprocess_run_dir')
    run_config = FullConfig.from_yaml(checkpoint_path / 'config.yaml')
    run_metrics = RunMetrics(**FullConfig._load_yaml(checkpoint_path / 'metrics.yaml'))
    style = PlotStyle.from_run(run_config, run_metrics).with_(
        path=PLOTS_ROOT, fig_filetype=FIGURE_FORMAT, dpi=300, vector_scaling=VECTOR_SCALING
    )

    # original_ts/ holds the raw, pre-discretization data as long-format per-series
    # CSVs -- there is no train_tsc.pkl.gz here (that is a learning-run artifact)
    train_tsc = load_original_ts_from_checkpoint(
        checkpoint_path,
        series_ids=[series_id],
        system_observables=run_config.preprocessing.system_observables,
    )

    print(f'\n\n\nProcess scenario {scenario}')
    # plot the ground truth density and velocity grids for a single visualization step
    true_time_values = TSCDataFrame.from_single_timeseries(train_tsc.loc[series_id]).time_values()
    vis_time_step = int(time_in_s / style.temporal_resolution)
    ground_truth_timestep = true_time_values[vis_time_step]
    _plot_ground_truth_grids(train_tsc, series_id, scenario, ground_truth_timestep, measurement_cell, style)